# Data Validation and Quality Inspection Module
Performs comprehensive data quality checks and reporting on the Cell2Cell dataset.

In [1]:
import pandas as pd
import numpy as np

REQUIRED_COLUMNS = [
    'REVENUE', 'MOU', 'RECCHRGE', 'DIRECTAS', 'OVERAGE', 'ROAM', 'CHANGEM',
    'CHANGER', 'DROPVCE', 'BLCKVCE', 'UNANSVCE', 'CUSTCARE', 'THREEWAY',
    'MOUREC', 'OUTCALLS', 'INCALLS', 'PEAKVCE', 'OPEAKVCE', 'DROPBLK',
    'CALLFWDV', 'CALLWAIT', 'CHURN', 'MONTHS', 'UNIQSUBS', 'ACTVSUBS',
    'CSA', 'PHONES', 'MODELS', 'EQPDAYS', 'CUSTOMER', 'AGE1', 'AGE2',
    'CHILDREN', 'CREDITA', 'CREDITAA', 'CREDITB', 'CREDITC', 'CREDITDE',
    'CREDITGY', 'CREDITZ', 'PRIZMRUR', 'PRIZMUB', 'PRIZMTWN', 'REFURB',
    'WEBCAP', 'TRUCK', 'RV', 'OCCPROF', 'OCCCLER', 'OCCCRFT', 'OCCSTUD',
    'OCCHMKR', 'OCCRET', 'OCCSELF', 'OWNRENT', 'MARRYUN', 'MARRYYES',
    'MARRYNO', 'MAILORD', 'MAILRES', 'MAILFLAG', 'TRAVEL', 'PCOWN',
    'CREDITCD', 'RETCALLS', 'RETACCPT', 'NEWCELLY', 'NEWCELLN', 'REFER',
    'INCMISS', 'INCOME', 'MCYCLE', 'CREDITAD', 'SETPRCM', 'SETPRC',
    'RETCALL', 'CALIBRAT', 'CHURNDEP'
]

def validate_dataset(df: pd.DataFrame) -> dict:
    
    report = {
        'total_rows': len(df),
        'total_cols': len(df.columns),
        'missing_columns': [],
        'missing_values': {},
        'duplicate_customers': 0,
        'anomalies': [],
        'status': 'PASSED'
    }

    # 1. Check Required Columns
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        report['missing_columns'] = missing_cols
        report['status'] = 'WARNING'

    # 2. Duplicate Check
    if 'CUSTOMER' in df.columns:
        dups = df['CUSTOMER'].duplicated().sum()
        report['duplicate_customers'] = int(dups)
        if dups > 0:
            report['anomalies'].append(f"Found {dups} duplicate CUSTOMER IDs.")

    # 3. Missing Value Check
    null_counts = df.isnull().sum()
    for col, count in null_counts[null_counts > 0].items():
        pct = round((count / len(df)) * 100, 2)
        report['missing_values'][col] = {'count': int(count), 'percentage': pct}

    # 4. Anomaly Checks
    if 'EQPDAYS' in df.columns:
        neg_eqp = (df['EQPDAYS'] < 0).sum()
        if neg_eqp > 0:
            report['anomalies'].append(f"Found {neg_eqp} negative values in EQPDAYS.")

    if 'MONTHS' in df.columns:
        invalid_months = (df['MONTHS'] <= 0).sum()
        if invalid_months > 0:
            report['anomalies'].append(f"Found {invalid_months} zero or negative values in MONTHS.")

    # 5. Check Target Distribution
    if 'CHURN' in df.columns:
        churn_dist = df['CHURN'].value_counts(normalize=True).to_dict()
        report['churn_distribution'] = {str(k): round(v, 4) for k, v in churn_dist.items()}

    return report

In [2]:
if __name__ == '__main__':
    try:
        df_sample = pd.read_csv('data/Sample Dataset.csv')
        val_report = validate_dataset(df_sample)
        print("=== DATA VALIDATION REPORT ===")
        print(f"Total Records: {val_report['total_rows']:,}")
        print(f"Total Columns: {val_report['total_cols']}")
        print(f"Status: {val_report['status']}")
        print("Churn Distribution:", val_report.get('churn_distribution'))
        print("Anomalies Detected:", val_report['anomalies'])
        print("Top Missing Columns:")
        for col, meta in list(val_report['missing_values'].items())[:5]:
            print(f"  - {col}: {meta['count']} ({meta['percentage']}%)")
    except FileNotFoundError:
        print("Sample Dataset.csv not found. Please provide the file to run testing.")

=== DATA VALIDATION REPORT ===
Total Records: 71,047
Total Columns: 78
Status: PASSED
Churn Distribution: {'0': 0.7099, '1': 0.2901}
Anomalies Detected: ['Found 107 negative values in EQPDAYS.']
Top Missing Columns:
  - REVENUE: 216 (0.3%)
  - MOU: 216 (0.3%)
  - RECCHRGE: 216 (0.3%)
  - DIRECTAS: 216 (0.3%)
  - OVERAGE: 216 (0.3%)
